# Dashboard de resultados de evaluación RAG

Este notebook carga los `*_eval.csv` y `*_summary.json` generados por `evaluacion_rag.ipynb` para:
- visualizar métricas por modelo,
- comparar retrieval/generación/robustez,
- revisar manualmente respuestas fallidas según reglas.


In [ ]:
!pip install -q -U pandas matplotlib seaborn plotly ipywidgets


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
pd.set_option('display.max_colwidth', 160)


In [ ]:
# CONFIG
ROOT = Path('/content/drive/MyDrive/NLP_PRACTICA/evaluacion_rag_runs')
ONLY_LATEST_PER_MODEL = True

# Reglas de fallo para revisión manual
JUDGE_FAIL_THRESHOLD = 3.0
FAITHFULNESS_FAIL_THRESHOLD = 0.5
SIM_FAIL_THRESHOLD = 0.45
RETRIEVAL_FAIL_THRESHOLD = 0.0  # hit@k <= 0 => fallo retrieval


In [ ]:
# CARGA DE ARCHIVOS
all_eval = sorted(ROOT.glob('**/*_eval.csv'))
all_summary = sorted(ROOT.glob('**/*_summary.json'))

if not all_eval:
    raise FileNotFoundError(f"No encontré *_eval.csv en {ROOT}")

print(f"Eval CSVs encontrados: {len(all_eval)}")
print(f"Summary JSON encontrados: {len(all_summary)}")

if ONLY_LATEST_PER_MODEL:
    latest = {}
    for p in all_eval:
        model = p.parent.name
        latest[model] = max(latest.get(model, p), p)
    eval_files = sorted(latest.values())
else:
    eval_files = all_eval

print("
CSVs usados:")
for p in eval_files:
    print(" -", p)


In [ ]:
# UNIÓN DE TODOS LOS CSVs
frames=[]
for p in eval_files:
    df = pd.read_csv(p)
    if 'model_alias' not in df.columns:
        df['model_alias'] = p.parent.name
    df['eval_file'] = str(p)
    frames.append(df)

full = pd.concat(frames, ignore_index=True)
print(full.shape)
display(full.head(2))


In [ ]:
# NORMALIZACIÓN DE TIPOS
num_cols = [
    'hit@k','mrr','context_precision@k','recall@k','context_recall@k',
    'semantic_similarity','cross_encoder_score','embedding_correct',
    'bigram_recall','trigram_recall','faithfulness_score','latency_sec',
    'judge_factualidad','judge_completitud','judge_alucinacion','judge_global',
    'abstention_pred','abstention_expected','abstention_correct'
]

for c in num_cols:
    if c in full.columns:
        full[c] = pd.to_numeric(full[c], errors='coerce')

if 'is_hard_negative' in full.columns:
    full['is_hard_negative'] = full['is_hard_negative'].fillna(False).astype(bool)


In [ ]:
# RESUMEN AGREGADO POR MODELO
agg = full.groupby('model_alias', dropna=False).agg(
    n=('question', 'count'),
    hit_k=('hit@k', 'mean'),
    mrr=('mrr', 'mean'),
    context_precision_k=('context_precision@k', 'mean'),
    recall_k=('recall@k', 'mean'),
    semantic_similarity=('semantic_similarity', 'mean'),
    embedding_acc=('embedding_correct', 'mean'),
    bigram_recall=('bigram_recall', 'mean'),
    trigram_recall=('trigram_recall', 'mean'),
    faithfulness=('faithfulness_score', 'mean'),
    judge_global=('judge_global', 'mean'),
    hard_negative_acc=('abstention_correct', lambda s: s[full.loc[s.index,'abstention_expected']==1].mean() if (full.loc[s.index,'abstention_expected']==1).any() else np.nan),
    latency_sec=('latency_sec', 'mean'),
).reset_index().sort_values('judge_global', ascending=False)

display(agg)


In [ ]:
# GRÁFICAS PRINCIPALES (matplotlib/seaborn)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

sns.barplot(data=agg, x='model_alias', y='hit_k', ax=axes[0,0])
axes[0,0].set_title('Hit@K medio por modelo')
axes[0,0].tick_params(axis='x', rotation=35)

sns.barplot(data=agg, x='model_alias', y='mrr', ax=axes[0,1])
axes[0,1].set_title('MRR medio por modelo')
axes[0,1].tick_params(axis='x', rotation=35)

sns.barplot(data=agg, x='model_alias', y='context_precision_k', ax=axes[0,2])
axes[0,2].set_title('Context Precision@K medio')
axes[0,2].tick_params(axis='x', rotation=35)

sns.barplot(data=agg, x='model_alias', y='semantic_similarity', ax=axes[1,0])
axes[1,0].set_title('Similitud semántica media')
axes[1,0].tick_params(axis='x', rotation=35)

sns.barplot(data=agg, x='model_alias', y='judge_global', ax=axes[1,1])
axes[1,1].set_title('Judge global medio')
axes[1,1].tick_params(axis='x', rotation=35)

sns.barplot(data=agg, x='model_alias', y='latency_sec', ax=axes[1,2])
axes[1,2].set_title('Latencia media (s)')
axes[1,2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
# MATRIZ DE CALOR DE MÉTRICAS NORMALIZADAS
metric_cols = ['hit_k','mrr','context_precision_k','recall_k','semantic_similarity','embedding_acc','faithfulness','judge_global','hard_negative_acc']
heat = agg.set_index('model_alias')[metric_cols]

plt.figure(figsize=(10,5))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='YlGnBu')
plt.title('Heatmap de métricas por modelo')
plt.show()


In [ ]:
# DASHBOARD INTERACTIVO (plotly): calidad vs coste
fig = px.scatter(
    agg,
    x='latency_sec',
    y='judge_global',
    size='hit_k',
    color='model_alias',
    hover_data=['mrr','context_precision_k','semantic_similarity','faithfulness','hard_negative_acc'],
    title='Trade-off calidad/coste (latencia vs judge_global)'
)
fig.show()


In [ ]:
# REGLAS DE FALLO (puedes ajustar umbrales arriba)
full['fail_retrieval'] = full['hit@k'].fillna(0) <= RETRIEVAL_FAIL_THRESHOLD
full['fail_semantic'] = full['semantic_similarity'].fillna(0) < SIM_FAIL_THRESHOLD
full['fail_faithfulness'] = full['faithfulness_score'].fillna(0) < FAITHFULNESS_FAIL_THRESHOLD
full['fail_judge'] = pd.to_numeric(full.get('judge_global', np.nan), errors='coerce').fillna(0) < JUDGE_FAIL_THRESHOLD
full['fail_hard_negative'] = (full['abstention_expected'].fillna(0)==1) & (full['abstention_correct'].fillna(0)==0)

full['any_fail'] = full[['fail_retrieval','fail_semantic','fail_faithfulness','fail_judge','fail_hard_negative']].any(axis=1)

fail_counts = full.groupby('model_alias')[['fail_retrieval','fail_semantic','fail_faithfulness','fail_judge','fail_hard_negative','any_fail']].mean().reset_index()
display(fail_counts)


In [ ]:
# LISTADO COMPLETO DE RESPUESTAS MALAS POR MODELO
cols_show = [
    'model_alias','idx','question','gold_answer','generated_answer',
    'hit@k','mrr','context_precision@k','recall@k',
    'semantic_similarity','cross_encoder_score','faithfulness_score','judge_global',
    'abstention_expected','abstention_pred','abstention_correct',
    'fail_retrieval','fail_semantic','fail_faithfulness','fail_judge','fail_hard_negative',
    'retrieved_chunks_json'
]

for model, d in full.groupby('model_alias'):
    bad = d[d['any_fail']].copy()
    bad = bad.sort_values(['fail_retrieval','fail_judge','fail_semantic','fail_faithfulness'], ascending=False)

    print('
' + '='*110)
    print(f"MODELO: {model} | respuestas marcadas como malas: {len(bad)} / {len(d)}")
    print('='*110)

    if len(bad)==0:
        print('No hay fallos según las reglas actuales.')
        continue

    display(bad[[c for c in cols_show if c in bad.columns]])


In [ ]:
# EXPORT opcional: CSVs con fallos para revisión manual fuera de notebook
OUT = ROOT / 'revision_fallos'
OUT.mkdir(parents=True, exist_ok=True)

for model, d in full.groupby('model_alias'):
    bad = d[d['any_fail']].copy()
    out = OUT / f'{model}_fallos.csv'
    bad.to_csv(out, index=False)

print(f"Fallos exportados en: {OUT}")
